# 🔧 Entrenamiento en la Nube con GPU — YOLOv8

**Proyecto:** Detección de defectos en piezas de fundición CNC
**Autor:** Brian Prado — Ingeniería Mecatrónica
**Modelo:** YOLOv8 (Ultralytics) · **Dataset:** casting-detection v10 (Roboflow, 8 clases)

Este notebook entrena el modelo **usando la GPU gratuita de Google Colab**, decenas de veces más rápida que una laptop sin GPU.

---

## ⚠️ PASO 0 — Activar la GPU (¡OBLIGATORIO!)

Antes de ejecutar cualquier celda:

1. Menú **`Entorno de ejecución`** → **`Cambiar tipo de entorno de ejecución`**.
2. En **`Acelerador por hardware`** elige **`GPU T4`**.
3. Pulsa **`Guardar`**.

Si omites este paso, todo correrá en CPU (lento). La siguiente celda confirma si la GPU está activa.

## Paso 1 — Verificar la GPU
Debe aparecer una tabla con una **Tesla T4** (o similar). Si no muestra ninguna GPU, repite el Paso 0.

In [ ]:
!nvidia-smi

## Paso 2 — Instalar dependencias
Instalamos Ultralytics (YOLOv8) y el cliente de Roboflow. Tarda ~1 minuto.

In [ ]:
!pip install -q ultralytics roboflow

## Paso 3 — Clonar tu repositorio de GitHub
Descarga tus scripts (`train.py`, `predict.py`, `download_dataset.py`) y la configuración del proyecto.

In [ ]:
!git clone https://github.com/BrianPrado-Dev/Vision-Artificial.git
%cd Vision-Artificial

## Paso 4 — Configurar tu API key de Roboflow (de forma segura)
Usamos `getpass` para que tu clave **no quede escrita ni guardada** en el notebook.
Al ejecutar la celda, **pega tu API key** en el cuadro que aparece y presiona Enter.
> ¿Dónde está tu key? En https://app.roboflow.com → Settings → API Keys.

In [ ]:
import os
from getpass import getpass

os.environ["ROBOFLOW_API_KEY"] = getpass("Pega tu API key de Roboflow y presiona Enter: ")
print("✅ API key configurada en esta sesión (no se guarda en el notebook).")

## Paso 5 — Descargar el dataset desde Roboflow
Tu script `download_dataset.py` toma la API key del paso anterior, descarga el dataset
y **corrige automáticamente las rutas** del `data.yaml` para que YOLO lo encuentre.

In [ ]:
!python scripts/download_dataset.py

## Paso 6 — Entrenar el modelo con GPU 🚀
Parámetros usados:
- `--model yolov8s.pt` → versión *small* (buen balance precisión/velocidad en GPU).
- `--epochs 50` → 50 pasadas al dataset (sube a 100 para más precisión).
- `--imgsz 640` → resolución de entrenamiento.
- `--batch 16` → imágenes por lote (la T4 lo soporta bien).
- `--device 0` → usa la **GPU**.

⏱️ En una T4, ~50 épocas suelen tardar **1–3 horas**. Mantén la pestaña abierta.

In [ ]:
!python scripts/train.py --data datasets/roboflow_fundicion/data.yaml --model yolov8s.pt --epochs 50 --imgsz 640 --batch 16 --device 0 --workers 2

## Paso 7 — Visualizar métricas del entrenamiento
Mostramos las gráficas que YOLO genera solo: curvas de pérdida/precisión, matriz de
confusión y curva Precisión-Recall. Son excelentes evidencias para tu informe.

In [ ]:
import glob, os
from IPython.display import Image, display

# Localiza la carpeta del último entrenamiento realizado.
carpetas = sorted(glob.glob('runs/entrenamiento/*'), key=os.path.getmtime)
run_dir = carpetas[-1]
print("📁 Resultados guardados en:", run_dir)

for nombre in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    ruta = os.path.join(run_dir, nombre)
    if os.path.exists(ruta):
        print("\n===== " + nombre + " =====")
        display(Image(filename=ruta))

## Paso 8 — Ver predicciones sobre imágenes de validación
YOLO guarda lotes de validación con las cajas predichas: útil para ver "qué tan bien ve".

In [ ]:
for nombre in ['val_batch0_pred.jpg', 'val_batch1_pred.jpg']:
    ruta = os.path.join(run_dir, nombre)
    if os.path.exists(ruta):
        display(Image(filename=ruta))

## Paso 9 — Inferencia sobre el conjunto de PRUEBA (test)
Ejecutamos tu `predict.py` con el mejor modelo (`best.pt`) sobre las imágenes de prueba
y mostramos algunos resultados con sus *bounding boxes*.

In [ ]:
!python scripts/predict.py --model {run_dir}/weights/best.pt --source datasets/roboflow_fundicion/test/images --conf 0.25

# Mostramos algunas predicciones de ejemplo
carpetas_pred = sorted(glob.glob('evidencias/predicciones*'), key=os.path.getmtime)
ejemplos = glob.glob(os.path.join(carpetas_pred[-1], '*.jpg'))[:5]
print("\nMostrando " + str(len(ejemplos)) + " predicciones de ejemplo:")
for p in ejemplos:
    display(Image(filename=p))

## Paso 10 — Descargar el modelo entrenado (`best.pt`)
Descarga el archivo a tu PC y colócalo en la carpeta **`models/`** de tu proyecto local
para usarlo con `predict.py`.

In [ ]:
from google.colab import files
print("Se descargará best.pt. Guárdalo en la carpeta models/ de tu proyecto.")
files.download(f"{run_dir}/weights/best.pt")

## (Opcional) Paso 11 — Guardar TODO en tu Google Drive
Copia la carpeta completa de resultados (pesos + gráficas) a tu Drive, por si Colab se desconecta.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r {run_dir} /content/drive/MyDrive/
print("✅ Resultados copiados a Google Drive (MyDrive).")

---
## ✅ Siguientes pasos y consejos

- **Guarda evidencias** para tu informe: descarga `results.png`, `confusion_matrix.png` y algunas predicciones.
- **Sube `best.pt` a tu repo**: colócalo en `models/` y haz commit/push (pesa ~22 MB).
- **¿Más precisión?** Aumenta `--epochs` a 100–150 o usa `--model yolov8m.pt`.
- **Las 8 clases del dataset:** `Casting_burr`, `Polished_casting`, `burr`, `crack`, `pit`, `scratch`, `strain`, `unpolished_casting`.
- **Colab se desconecta** tras inactividad; mantén la pestaña abierta durante el entrenamiento.

> 💡 Tip: si Colab limita la GPU, reduce `--epochs` o `--imgsz` (p. ej. 416).